In [ ]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import contextily as cx
from matplotlib.colors import ListedColormap
import mapclassify



# Load missouri shapefiles
mo_gdf = gpd.read_file('./tract_shapefiles/tl_2023_mo_tract/tl_2023_29_tract.shp')
#gdf = gdf.set_crs('EPSG:4269', allow_override=True)
# Make sure the CRS is in a projected coordinate system (e.g., UTM) for accurate area calculation
if mo_gdf.crs.is_geographic:
    mo_gdf = mo_gdf.to_crs(epsg=26915)  # UTM zone 15N, adjust if necessary

In [ ]:
mo_roads = gpd.read_file('./tl_2019_29_prisecroads/tl_2019_29_prisecroads.shp')
stl_bbox = (-90.45, 38.5, -90.1, 38.9)  # Adjust if necessary

stl_secondary_roads = mo_roads.cx[stl_bbox[0]:stl_bbox[2], stl_bbox[1]:stl_bbox[3]]
stl_secondary_roads = stl_secondary_roads.to_crs(epsg=26915)

stl_parks = gpd.read_file('./parks/parks.shp')
stl_parks = stl_parks.to_crs(epsg=26915)

In [ ]:
import geopandas as gpd
import pandas as pd

# 1. Read & clean scooter stats (unchanged)
stats = (
    pd.read_csv(
        './stl/St-Louis-routes-data-for-all-vehicles-in-all-time.csv'
    )
    .rename(columns={
        'Segment Geometry ID': 'geometryId',
        'Count of Matched Trips': 'scooter_rides'
    })
)
stats['scooter_rides'] = (
    stats['scooter_rides']
    .astype(str)
    .str.replace(',', '', regex=False)
    .pipe(pd.to_numeric, errors='coerce')
    .fillna(0)
    .astype(int)
)

# 2. Load scooter GeoJSON, merge counts, compute full‐segment lengths
scooter_geo = (
    gpd.read_file('./stl/St-Louis-routes-data-for-all-vehicles-in-all-time.geojson')
    .to_crs(epsg=26915)
    .merge(stats[['geometryId','scooter_rides']], on='geometryId', how='left')
)
# drop any null rides
scooter_geo['scooter_rides'] = scooter_geo['scooter_rides'].fillna(0)
# compute each segment’s total length
scooter_geo['seg_length'] = scooter_geo.geometry.length

# 3. Load & filter tracts
tracts = (
    gpd.read_file('./tract_shapefiles/tl_2023_mo_tract/tl_2023_29_tract.shp')
    .to_crs(epsg=26915)
    .query("COUNTYFP == '510'")
    .rename(columns={'TRACTCE':'tract_id'})
    [['tract_id','geometry']]
)

# 4. Overlay intersection → each piece is segment‐portion inside one tract
intersected = gpd.overlay(
    scooter_geo[['geometry','scooter_rides','seg_length']],
    tracts,
    how='intersection'
)
# compute how much of the original segment is in that piece
intersected['piece_length'] = intersected.geometry.length
intersected['frac_in_tract'] = (
    intersected['piece_length'] / intersected['seg_length']
)
# weight the rides by that fraction
intersected['weighted_rides'] = (
    intersected['scooter_rides'] * intersected['frac_in_tract']
)

# 5. Sum up per‐tract weighted rides
scooter_by_tract = (
    intersected
      .groupby('tract_id', as_index=False)['weighted_rides']
      .sum()
      .rename(columns={'weighted_rides':'scooter_rides'})
)

# 6. Merge back to tracts and compute densities
plot_gdf = tracts.merge(scooter_by_tract, on='tract_id', how='left')
plot_gdf['scooter_rides'] = plot_gdf['scooter_rides'].fillna(0)
# average per‐day
plot_gdf['scooter_per_day'] = plot_gdf['scooter_rides'] / 2163
# area in square miles
plot_gdf['area_sqmi'] = plot_gdf.geometry.area / (5280**2)
# density per day per sq.mi
plot_gdf['scooter_density_per_day'] = (
    plot_gdf['scooter_per_day'] / plot_gdf['area_sqmi']
)




In [ ]:
plot_gdf['scooter_per_day'].describe()


In [ ]:
plt.hist(plot_gdf['scooter_per_day'])

In [ ]:
top4 = plot_gdf.nlargest(4, 'scooter_per_day')[['tract_id','scooter_per_day']]
print(top4)


In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
import matplotlib.patches as mpatches
import matplotlib.lines as mlines

# 5. Plot density, parks, roads, and tract borders
fig, ax = plt.subplots(figsize=(10, 8))

# Density choropleth
plot_gdf.plot(
    column='scooter_density_per_day',
    cmap='Blues',
    ax=ax,
    legend=True,
    legend_kwds={'label': 'Segment Traversals per sq.mi', 'shrink': 0.5},
    missing_kwds={'color': 'lightgrey'}
)

# Overlay parks and roads
#stl_parks.plot(ax=ax, color='lightgreen', alpha=0.5, edgecolor='none')
#stl_secondary_roads.plot(ax=ax, color='grey', linewidth=0.5)

# Census tract boundaries
tracts.boundary.plot(ax=ax, color='black', linewidth=0.2)

# Legends for map features
parks_patch = mpatches.Patch(color='lightgreen', label='Parks')
roads_line = mlines.Line2D([], [], color='grey', linewidth=1, label='Secondary Roads')
tracts_line = mlines.Line2D([], [], color='black', linewidth=1, label='Tract Boundaries')

ax.legend(
    handles=[parks_patch, roads_line, tracts_line],
    title='Map Features',
    loc='upper left',
    facecolor='white',
    framealpha=1.0

)

ax.set_title('Scooter Ride Density by Census Tract\nSt. Louis City', fontsize=14)
#ax.set_axis_off()

xmin, xmax = 725000, 746500  
ymin, ymax = 4265000, 4297000

ax.set_xlim([xmin, xmax])
ax.set_ylim([ymin, ymax])

# Add basemap
cx.add_basemap(ax, crs=tracts.crs, source=cx.providers.CartoDB.Positron, zoom=12)

plt.tight_layout()
plt.show()

In [ ]:
def fmt_interval(iv, precision=1):
    """Return 'left – right' with the given decimal precision."""
    left  = f"{iv.left:.{precision}f}"
    right = f"{iv.right:.{precision}f}"
    return f"{left} – {right}"

In [ ]:
import numpy as np
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.colors import ListedColormap
import contextily as cx
import mapclassify           # only for the nice legend title formatting

# ------------------------------------------------------------------
# 1.  Make the quantile bins (duplicates may be dropped!)
# ------------------------------------------------------------------
plot_gdf['density_bin'] = pd.qcut(
    plot_gdf['scooter_density_per_day']
        .fillna(0)
        .astype(float),          # be sure it’s numeric
    q=10,
    duplicates='drop'
)

breaks     = plot_gdf['density_bin'].cat.categories
n_classes  = len(breaks)

# ------------------------------------------------------------------
# 2.  Build exactly n_classes colours from a sequential “hot” palette
# ------------------------------------------------------------------
cmap_base  = plt.cm.Blues      # or any perceptually-ordered palette
numbers = np.logspace(0, 1, n_classes)
numbers = (numbers - 1) / (n_classes - 1)
custom_cmap = cmap_base(numbers)
discrete_cmap = ListedColormap(custom_cmap)

# ------------------------------------------------------------------
# 3.  Plot
# ------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 8))

# tracts coloured by bin
plot_gdf.plot(
    column='density_bin',
    cmap=discrete_cmap,
    linewidth=0.1, edgecolor='black',
    legend=False,
    ax=ax
)

# overlay context layers
stl_parks.plot(ax=ax, color='lightgreen', alpha=0.5, edgecolor='none')
stl_secondary_roads.plot(ax=ax, color='grey', linewidth=0.5)

# ------------------------------------------------------------------
# 4.  Build a matching legend
# ------------------------------------------------------------------
patches = []
for idx, interval in enumerate(breaks):
    label = fmt_interval(interval, precision=1)   # ← replace call here
    patches.append(mpatches.Patch(color=custom_cmap[idx], label=label))

leg_bins = ax.legend(
    handles=patches,
    title="Scooter density\n(rides / sq mi, quantiles)",
    loc='upper left',
    frameon=True, facecolor='white'
)

# map-feature legend
#parks_patch = mpatches.Patch(color='lightgreen', label='Parks')
roads_line  = mlines.Line2D([], [], color='grey', linewidth=0.8, label='Secondary roads')
leg_feats = ax.legend(
    handles=[parks_patch, roads_line],
    title="Map features",
    loc='upper right',
    frameon=True, facecolor='white'
)

ax.add_artist(leg_bins)   # ensure the quantile legend stays

# ------------------------------------------------------------------
# 5.  Final cosmetics
# ------------------------------------------------------------------
ax.set_axis_off()
ax.set_title("Scooter-ride density quantiles by census tract\nSt Louis City", fontsize=14)
xmin, xmax = 725000, 746500  
ymin, ymax = 4265000, 4297000

ax.set_xlim([xmin, xmax])
ax.set_ylim([ymin, ymax])
cx.add_basemap(ax, crs=plot_gdf.crs, source=cx.providers.CartoDB.Positron, zoom=12)

plt.tight_layout()
plt.show()


In [ ]:
stl_predictions = pd.read_csv('./stl/stl_z_preds_2.csv')


In [ ]:
stl_predictions = stl_predictions.rename(columns={'geoid': 'GEOID'})
stl_predictions['GEOID'] = stl_predictions['GEOID'].astype(str)
predictions_gdf = mo_gdf.merge(stl_predictions, on='GEOID')
predictions_gdf['predicted_density'] = (predictions_gdf['predicted_trips_z']) / (predictions_gdf['ALAND'] / 2_589_988.11)
mu, sigma = predictions_gdf['predicted_density'].mean(), predictions_gdf['predicted_density'].std()
predictions_gdf['predicted_density_z'] = (predictions_gdf['predicted_density'] - mu) / sigma


In [ ]:
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import contextily as cx

fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Plot bike trip predictions with automatic legend for the ride counts
predictions_gdf.plot(column='predicted_density', cmap='YlGnBu', legend=True, 
                legend_kwds={'label': "Predicted Number of Bike Trips"},
                edgecolor='black', linewidth=0.5, ax=ax)

# positive_predictions_gdf['trips_per_area'] = positive_predictions_gdf['trips_per_day'] / (positive_predictions_gdf['ALAND'] / 2_589_988.11)
# positive_predictions_gdf.plot(column='trips_per_area', cmap='YlGnBu', legend=True, 
#                 legend_kwds={'label': "Predicted Number of Bike Trips"},
#                 edgecolor='black', linewidth=0.5, ax=ax)

# Add title and remove axes for clarity
ax.set_title("BikeShare predictions", fontsize=14)
ax.set_axis_off()

# Plot major roads and parks
stl_secondary_roads.plot(ax=ax, color='whitesmoke', linewidth=0.5, alpha=1.0, label='_nolabel_')
stl_parks.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.7, label='_nolabel_')

# Create custom legend handles
bike_legend_patch = mpatches.Patch(color='gray', label='St. Louis City Parks')
roads_legend_line = mlines.Line2D([], [], color='lightgrey', linewidth=1.5, label='Major Roads')

# Add custom legend for parks and roads separately
ax.legend(handles=[bike_legend_patch, roads_legend_line], loc='upper right', title="Map Features")

# Set title and turn off axis
plt.axis('off')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
bincount = 10
# Classify data into quartiles explicitly
classifier = mapclassify.Quantiles(predictions_gdf['predicted_density_z'], k=bincount)
predictions_gdf['quartile'] = classifier.yb

# Define color map
cmap = plt.cm.YlGnBu
# Evenly spaced numbers from 0 to 1
numbers = np.logspace(0, 1, bincount)
numbers = (numbers - 1) / 9
#numbers = np.linspace(0,1,10)
colors = cmap(numbers)  # 4 colors for quartiles
custom_cmap = ListedColormap(colors)

# Plot quartile data
predictions_gdf.plot(
    column='quartile',
    cmap=custom_cmap,
    alpha=0.9,
    edgecolor='black',
    linewidth=0.1,
    ax=ax,
    legend=False
)

# # Plot roads and parks
stl_secondary_roads.plot(ax=ax, color='whitesmoke', linewidth=0.5, alpha=1.0)
stl_parks.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.7)

# Custom legend for quartiles
quartile_labels = []
bins = classifier.bins
quartile_labels.append(f'{predictions_gdf["predicted_density_z"].min():.2f} – {bins[0]:.2f}')
for i in range(1, len(bins)):
    quartile_labels.append(f'{bins[i-1]:.2f} – {bins[i]:.2f}')

quartile_patches = [
    mpatches.Patch(color=colors[i], label=quartile_labels[i]) for i in range(bincount)
]

# # Custom legend for map features
parks_patch = mpatches.Patch(color='gray', label='St. Louis City Parks')
roads_line = mlines.Line2D([], [], color='whitesmoke', linewidth=1.5, label='Major Roads')
parks_patch.set_zorder(9998)
roads_line.set_zorder(9998)

# # Add legends separately with clear distinction
leg1 = ax.legend(
    handles=quartile_patches,
    title="Density Quantiles \n (Z-scoredrides/day)/sq.mi",
    loc='upper left',
    facecolor='lightgray',  # or use fancybox=True
    framealpha=1.0
)
# Force legend on top
leg1.set_zorder(9999)
ax.add_artist(leg1)  # Add first legend explicitly back to plot

ax.legend(handles=[parks_patch, roads_line], title="Map Features", loc='upper right',
          facecolor='lightgray',   # Legend background
          )


# Set titles and remove axes
ax.set_title("Ensemble ElasticNet Predictions for Bike Share Use in St. Louis \n (Colored by Quantiles, 10 bins)", fontsize=12)
ax.set_axis_off()
cx.add_basemap(ax, crs=26915, source=cx.providers.CartoDB.Positron, zoom=12)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
from matplotlib.colors import ListedColormap
import contextily as cx

# 1) compute each tract’s percentile rank of predicted_density_z
predictions_gdf['pct_rank'] = (
    predictions_gdf['predicted_density_z']
      .rank(pct=True, method='first')
)

# 2) cut into the five buckets you asked for
bins = [0.0, 0.50, 0.75, 0.90, 0.95, 1.00]
labels = ['0–50%', '50–75%', '75–90%', '90–95%', '95–100%']
predictions_gdf['demand_bin'] = pd.cut(
    predictions_gdf['pct_rank'],
    bins=bins,
    labels=labels,
    include_lowest=True
)

# 3) pick a “hot” colormap with 5 steps
cmap = plt.cm.Blues
colors = [cmap(i/4) for i in range(5)]
custom_cmap = ListedColormap(colors)

# 4) plot
fig, ax = plt.subplots(figsize=(10,8))
predictions_gdf.plot(
    column='demand_bin',
    categorical=True,
    cmap=custom_cmap,
    linewidth=0.1,
    edgecolor='black',
    legend=False,
    ax=ax
)

# overlay roads/parks (if you have them)
stl_secondary_roads.plot(ax=ax, color='gray', linewidth=0.5)
#stl_parks          .plot(ax=ax, color='lightgreen', linewidth=0.5, alpha = 0.5)

# 5) build a custom legend
patches = [
    mpatches.Patch(color=colors[i], label=labels[i])
    for i in range(len(labels))
]
ax.legend(
    handles=patches,
    title="Predicted Demand\nPercentile bins",
    loc='upper left',
    frameon=True,
    facecolor='white'
)

xmin, xmax = 725000, 746500  
ymin, ymax = 4265000, 4297000

ax.set_xlim([xmin, xmax])
ax.set_ylim([ymin, ymax])

ax.set_title("Where Predicted Bike-share Demand Is Highest (St. Louis)")
ax.set_axis_off()
cx.add_basemap(ax, crs=predictions_gdf.crs, source=cx.providers.CartoDB.Positron)
plt.tight_layout()
plt.show()


In [ ]:
predictions_gdf.columns

In [ ]:
plot_gdf.columns

In [ ]:
# --- restrict to City tracts (if you haven't already) ---
city_mask = predictions_gdf['COUNTYFP'] == '510'

# --- take only the density you care about (e.g. predicted_density_z or raw density) ---
bike_den = predictions_gdf.loc[city_mask, 'predicted_density']

# --- compute city‐only mean/std ---
μ_city = bike_den.mean()
σ_city = bike_den.std(ddof=0)

# --- overwrite (or make a new) z‐score column on just those city tracts ---
predictions_gdf.loc[city_mask, 'pred_density_city_z'] = (bike_den - μ_city) / σ_city

predictions_gdf['tract_id'] = (
    predictions_gdf['GEOID']
    .astype(str)
    .str[-6:]    # keep only the last 6 characters
)



In [ ]:
import numpy as np

# 1. Compute mean & standard deviation over the City tracts
μ = plot_gdf['scooter_density_per_day'].mean()
σ = plot_gdf['scooter_density_per_day'].std(ddof=0)
if σ == 0:
    σ = 1.0

# 2. Create the z column
plot_gdf['scooter_density_z'] = (
    plot_gdf['scooter_density_per_day'] - μ
) / σ


In [ ]:
predictions_gdf.columns

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
import numpy as np

# -----------------------------------------------------------------------------
# 1) Merge scooter + bike‐share z‐scores into one gdf
# -----------------------------------------------------------------------------
# assume:
#   - plot_gdf has columns ['tract_id','scooter_density_z','geometry']
#   - predictions_gdf has columns ['tract_id','predicted_density_z','geometry']
# Make sure they share the same index/key:
gap_gdf = predictions_gdf[['tract_id','geometry','predicted_density_z']] \
              .merge(
                  plot_gdf[['tract_id','scooter_density_z']],
                  on='tract_id', how='inner'
              )
# now gap_gdf has geometry, predicted_density_z, and scooter_density_z 

gap_gdf = gpd.GeoDataFrame(gap_gdf, geometry='geometry', crs=plot_gdf.crs)
print(gap_gdf.shape)
# -----------------------------------------------------------------------------
# 2) Compute the gap z‐score
# -----------------------------------------------------------------------------
gap_gdf['gap_z'] = gap_gdf['predicted_density_z'] - gap_gdf['scooter_density_z']

# for a symmetric color scale:
m = np.nanmax(np.abs(gap_gdf['gap_z']))
vmin, vmax = -m, +m

# -----------------------------------------------------------------------------
# 3) Plot with a diverging “hot/cold” map
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10,10))

gap_gdf.plot(
    column    = 'gap_z',
    cmap      = 'RdBu_r',    # blue=cold, red=hot
    vmin      = vmin,
    vmax      = vmax,
    legend    = True,
    legend_kwds = {
       'label': "Bike-minus-Scooter Demand (z-score gap)",
       'shrink': 0.6
    },
    edgecolor = 'lightgrey',
    linewidth  = 0.2,
    ax         = ax
)

# add contextily basemap
cx.add_basemap(
    ax,
    crs    = gap_gdf.crs,
    source = cx.providers.CartoDB.Positron,
    zoom   = 12
)

# polish
ax.set_axis_off()
ax.set_title("Unmet Bike-Share Demand vs. Scooter Use\n(positive=hotspots for new bike stations)", 
             fontsize=14)

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import contextily as cx
from mpl_toolkits.axes_grid1 import make_axes_locatable

# 1) Grab your Δ-Z series & global max
g = gap_gdf['gap_z']
m = np.nanmax(np.abs(g))

# 2) Choose a larger linear threshold & milder log base
linthresh = 1.0    # ±1σ stays purely linear
linscale  = 0.5    # that ±1σ band occupies twice the bar height
base      = 2      # use log₂ instead of log₁₀ for softer compression

norm = mcolors.SymLogNorm(
    linthresh=linthresh,
    linscale =linscale,
    base     =base,
    vmin     =-m,
    vmax     =+m
)

# 3) Plot
fig, ax = plt.subplots(figsize=(8,11))
gap_gdf.plot(
    column='gap_z',
    cmap=plt.cm.RdBu_r,
    norm=norm,
    linewidth=0.2, edgecolor='k',
    ax=ax
)
ax.set_axis_off()
ax.set_title("Difference between Predicted Docked Bike Share Demand\n and Current Dockless Scooter Use for St. Louis City", fontsize=14)

# 4) Colourbar
divider = make_axes_locatable(ax)
cax = divider.append_axes("right", size="4%", pad=0.05)
sm = plt.cm.ScalarMappable(norm=norm, cmap=plt.cm.RdBu_r)
sm._A = []
cbar = fig.colorbar(sm, cax=cax)
ticks = np.array([0,1,2,4,8,16])
ticks = np.unique(np.concatenate([-ticks, ticks]))
ticks = ticks[np.abs(ticks) <= m]
cbar.set_ticks(ticks)
cbar.set_ticklabels([f"{t:+.0f}" for t in ticks])
cbar.ax.set_ylabel("Difference in Z-score (Bike Share minus Scooter) with mild log compression of scale", rotation=270, labelpad=15)

# 5) Basemap
cx.add_basemap(ax, crs=gap_gdf.crs, source=cx.providers.CartoDB.Positron, zoom=12)
plt.tight_layout()
plt.show()


In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily as cx
import numpy as np

# -----------------------------------------------------------------------------
# 1) Merge scooter + bike data (unchanged)
# -----------------------------------------------------------------------------
gap_gdf = predictions_gdf[['tract_id','geometry','predicted_density']].merge(
              plot_gdf[['tract_id','scooter_density']],
              on='tract_id', how='inner'
          )
gap_gdf = gpd.GeoDataFrame(gap_gdf, geometry='geometry', crs=plot_gdf.crs)

# -----------------------------------------------------------------------------
# 2) Convert each variable to a percentile rank
# -----------------------------------------------------------------------------
for col in ['predicted_density', 'scooter_density']:
    gap_gdf[col + '_pct'] = gap_gdf[col].rank(pct=True)   # 0–1

# Percentile‑rank gap: +1 → bike higher, –1 → scooter higher
gap_gdf['gap_pct'] = gap_gdf['predicted_density_pct'] - gap_gdf['scooter_density_pct']

# symmetric colour limits (–1 … +1)
vmin, vmax = -1, 1        # percentile differences always fall in this range

# -----------------------------------------------------------------------------
# 3) Plot
# -----------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(10, 10))

gap_gdf.plot(
    column      = 'gap_pct',
    cmap        = 'RdBu_r',
    vmin        = vmin,
    vmax        = vmax,
    legend      = True,
    legend_kwds = dict(
        label   = "Bike percentile – Scooter percentile",
        shrink  = 0.6,
        format  = '%.2f'   # show –1.00 … 1.00
    ),
    edgecolor   = 'lightgrey',
    linewidth   = 0.2,
    ax          = ax
)

cx.add_basemap(
    ax,
    crs   = gap_gdf.crs,
    source= cx.providers.CartoDB.Positron,
    zoom  = 12
)

ax.set_axis_off()
ax.set_title(
    "Bike‑Share vs. Scooter Use (percentile gap)\n"
    "positive = bike demand > scooter use",
    fontsize=14
)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import contextily as cx

# 1) build a 3-way class from your pred_density_z:
predictions_gdf['density_class'] = pd.cut(
    predictions_gdf['predicted_density_z'],
    bins=[-np.inf, 1.0, 2.0, np.inf],
    labels=['z ≤ 1', '1 < z ≤ 2', 'z > 2']
)

# 2) pick a ‘hot’ palette – here I use matplotlib’s “hot” stretched over 3 colors

#hot_colors = plt.cm.YlGnBu(np.linspace(0.1, 1.0, 3))
hot_colors = plt.cm.hot(np.linspace(0.99, 0.3, 3))

hot_cmap    = ListedColormap(hot_colors)
#hot_cmap = plt.cm.YlGnBu

# 3) plot
fig, ax = plt.subplots(figsize=(10, 8))
predictions_gdf.plot(
    column='density_class',
    categorical=True,
    cmap=hot_cmap,
    edgecolor='black',
    linewidth=0.2,
    legend=True,
    legend_kwds={
      'title': 'Predicted Demand\nDensity (z-score)',
      'loc': 'upper left',
      'frameon': True,
      'framealpha': 1.0
    },
    ax=ax
)

# 4) add your roads & parks layers back in underneath
stl_secondary_roads.plot(ax=ax, color='whitesmoke', linewidth=0.5, alpha=1.0)
stl_parks.plot(ax=ax, color='gray',        linewidth=0.5, alpha=0.7)

# 5) basemap
cx.add_basemap(ax, crs=predictions_gdf.crs.to_string(),
               source=cx.providers.CartoDB.Positron, zoom=12)

ax.set_axis_off()
ax.set_title("St. Louis Bike-Share Hotspots\n(predicted_density_z >1 and >2 highlighted)",
             fontsize=14)
plt.tight_layout()
plt.show()



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# 1) compute the threshold values
pred = predictions_gdf['predicted_density_z']
th95 = pred.quantile(0.95)
th90 = pred.quantile(0.90)
th75 = pred.quantile(0.75)

# 2) assign each tract a category
#    note: order matters—Top 5% must be checked first
conds = [
    pred >= th95,
    (pred >= th90) & (pred < th95),
    (pred >= th75) & (pred < th90),
]
choices = ['Top 5%', 'Top 10%', 'Top 25%']
predictions_gdf['heat_tier'] = np.select(conds, choices, default='Other')

# 3) pick a small discrete colormap
cmap = ListedColormap(['lightgray','yellow','orange','red'])
# make sure categories plot in the right order:
cat_order = ['Other','Top 25%','Top 10%','Top 5%']

fig, ax = plt.subplots(figsize=(10,10))
# pass `categorical=True` so it doesn’t try to interpolate colors
predictions_gdf.plot(
    column='heat_tier',
    categorical=True,
    categories=cat_order,
    cmap=cmap,
    linewidth=0.2,
    edgecolor='white',
    ax=ax,
    legend=True,
    legend_kwds={'title':'Pred. density\n(by Z‐score percentile)'}
)

ax.set_axis_off()
ax.set_title("Bike‐share demand density: Top 5%, 10%, 25%")
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# grab your z-scores and drop NaNs
data = predictions_gdf['predicted_density_z'].dropna()

# 1) define a fine bin‐width, e.g. 0.1 z‐score units
bin_width = 0.1
bins = np.arange(data.min(), data.max() + bin_width, bin_width)

# 2) compute KDE
kde = gaussian_kde(data)

# 3) set up the plot
fig, ax = plt.subplots(figsize=(8, 5))

# histogram as density
ax.hist(
    data,
    bins=bins,
    density=True,
    alpha=0.6,
    color='steelblue',
    edgecolor='white'
)

# overlay KDE
x_vals = np.linspace(data.min(), data.max(), 500)
ax.plot(
    x_vals,
    kde(x_vals),
    color='black',
    linewidth=1.5,
    label='KDE'
)

# 4) add vertical lines at mean and ±1σ
mean = data.mean()
std  = data.std()
for offset, ls, label in [ (0,      '--', 'Mean'),
                           ( std,   ':',  '+1σ'),
                           (-std,   ':',  '–1σ') ]:
    ax.axvline(
        mean + offset,
        color='red',
        linestyle=ls,
        linewidth=1,
        label=label if offset==0 else None  # only label the mean once
    )

# formatting
ax.set_title('Distribution of Predicted Density Z-Scores', fontsize=14)
ax.set_xlabel('Z-score')
ax.set_ylabel('Probability Density')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()


In [ ]:
predictions_gdf[predictions_gdf['predicted_density_z'] ==predictions_gdf['predicted_density_z'].max()]

In [ ]:
# ── predictions_gdf  → has   geoid, predicted_trips_z, predicted_density, …
# ── plot_gdf         → has   geoid, scooter_rides,    scooter_density, …
plot_gdf = plot_gdf.rename(columns={'geoid': 'GEOID'})

both = (
    predictions_gdf[['GEOID', 'predicted_trips_z', 'predicted_density',
                     'predicted_density_z']]
    .merge(
        plot_gdf[['GEOID', 'scooter_rides', 'scooter_density']],
        on='GEOID', how='inner', suffixes=('', '_scoot')
    )
)
print(f"{both.shape[0]} tracts with info from **both** sources")


In [ ]:
from scipy import stats

for col in ['scooter_rides', 'scooter_density']:
    zc = f"{col}_z"
    both[zc] = stats.zscore(both[col].astype(float), nan_policy='omit')


In [ ]:
both['gap_trips_z']    = both['predicted_trips_z']    - both['scooter_rides_z']
both['gap_density_z']  = both['predicted_density_z']  - both['scooter_density_z']


In [ ]:
corr = both[['predicted_trips_z', 'scooter_rides_z']].corr().iloc[0,1]
print(f"Pearson r (bike model vs scooter rides) = {corr:0.3f}")

print("\nTop-10 unmet demand tracts (gap_trips_z):")
print(both.nlargest(10, 'gap_trips_z')[['GEOID','gap_trips_z']])


In [ ]:
import seaborn as sns, matplotlib.pyplot as plt

sns.regplot(
    data=both, x='scooter_rides_z', y='predicted_trips_z',
    scatter_kws=dict(alpha=.6, edgecolor='none')
)
plt.axline((0,0),(1,1), color='k', lw=.8, ls='--')   # 1-to-1 line
plt.xlabel("Scooter rides  (Z)")
plt.ylabel("Predicted docked-bike trips  (Z)")
plt.title("Docked-bike forecast vs observed scooter activity by tract")
plt.show()


In [ ]:
import geopandas as gpd
import mapclassify
import matplotlib.pyplot as plt
import contextily as cx
import numpy as np
import matplotlib.patches as mpatches

# 1) Rebuild gap_gdf as a GeoDataFrame
gap_gdf = both.merge(
    predictions_gdf[['GEOID','geometry']],
    on='GEOID', how='left'
)
gap_gdf = gpd.GeoDataFrame(gap_gdf,
                           geometry='geometry',
                           crs=predictions_gdf.crs)

# reproject to Web-Mercator for basemap
gap_3857 = gap_gdf.to_crs(epsg=3857)

# 2a) Option A: quantile classification (7 classes)
classifier = mapclassify.Quantiles(gap_3857['gap_density_z'], k=7)
gap_3857['bin'] = classifier.yb   # integers 0…6

# pick a “hot” palette of length 7
hot_colors = plt.cm.hot(np.linspace(0.99,0.3,7))
hot_cmap    = ListedColormap(hot_colors)


fig, ax = plt.subplots(figsize=(10,8))
gap_3857.plot(
    column='bin',
    cmap=hot_cmap,
    categorical=True,
    linewidth=0.1,
    edgecolor='black',
    legend=False,
    ax=ax
)
cx.add_basemap(ax,
               source=cx.providers.CartoDB.Positron,
               crs=gap_3857.crs.to_string())

# build a custom legend
patches = []
labels  = []
prev = gap_3857['gap_density_z'].min()
for i, upper in enumerate(classifier.bins):
    labels.append(f"{prev:.2f} → {upper:.2f}")
    patches.append(mpatches.Patch(color=hot_colors[i], label=labels[-1]))
    prev = upper

ax.legend(handles=patches,
          title="Unmet Demand (quantiles of z-score)",
          loc="upper left",
          framealpha=1)

ax.set_axis_off()
ax.set_title("Hotspots of Unmet Dock-Bike vs Scooter Demand")
plt.show()


In [ ]:
import geopandas as gpd
mo_gdf = gpd.read_file('./tl_2020_mo_tract/tl_2020_29_tract.shp')
#gdf = gdf.set_crs('EPSG:4269', allow_override=True)
# Make sure the CRS is in a projected coordinate system (e.g., UTM) for accurate area calculation
if mo_gdf.crs.is_geographic:
    mo_gdf = mo_gdf.to_crs(epsg=26915)  # UTM zone 15N, adjust if necessary

# Load stl transporation data
census_files = [
    "./census_data/S0101_2023.csv",
    "./census_data/S0601_2023.csv",
    "./census_data/S0701_2023.csv",
    "./census_data/S0801_2023.csv",
    "./census_data/S0802_2023.csv",
    "./census_data/DP03_2023.csv",
]

variable_meanings = {}

merged_stl_data = pd.read_csv(census_files[0])
merged_stl_data['GEOID'] = merged_stl_data['GEO_ID'].str.split('US').str[1]  # Extract GEOID from GEO_ID
merged_stl_data = merged_stl_data.drop(columns=['GEO_ID'])  # Drop redundant column
print(merged_stl_data['GEOID'])

meanings = merged_stl_data.iloc[0].to_dict()  # Convert the first row to a dictionary
variable_meanings.update(meanings)  # Update the global dictionary of meanings
merged_stl_data = merged_stl_data.iloc[1:].reset_index(drop=True)

# Sequentially merge the remaining CSVs
for file in stl_csv_files[1:]:
    df = pd.read_csv(file)
    df['GEOID'] = df['GEO_ID'].str.split('US').str[1]  # Extract GEOID from GEO_ID
    df = df.drop(columns=['GEO_ID'])  # Drop redundant column
    df = df.drop(columns=['NAME'])
    meanings = df.iloc[0].to_dict()  # Convert the first row to a dictionary
    variable_meanings.update(meanings)  # Update the global dictionary of meanings
    df = df.iloc[1:].reset_index(drop=True)

    merged_stl_data = pd.merge(merged_stl_data, df, on='GEOID', how='left')  # Merge on GEOID

merged_df = merged_stl_data

# Ensure all object columns are converted to numeric where possible
for col in merged_df.select_dtypes(include=['object']).columns:
    merged_df[col] = pd.to_numeric(merged_df[col], errors='coerce')

In [ ]:
mo_gdf['GEOID'] = mo_gdf['GEOID'].astype(str)
merged_df['GEOID'] = merged_df['GEOID'].astype(str)
merged_gdf = mo_gdf.merge(merged_df, on='GEOID')

In [ ]:
stl_predictions = pd.read_csv('./stl_predictions.csv')
print(stl_predictions.columns)
print(stl_predictions['trips_per_day'].describe())

stl_predictions['GEOID'] = stl_predictions['GEOID'].astype(str)
predictions_gdf = mo_gdf.merge(stl_predictions, on='GEOID')
print(predictions_gdf['trips_per_day'].describe())
print(predictions_gdf.loc[predictions_gdf['trips_per_day']>0, 'trips_per_day'].describe())
positive_predictions_gdf = predictions_gdf[predictions_gdf['trips_per_day']>0]
positive_predictions_gdf

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import contextily as cx

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
predictions_gdf['trips_per_area'] = (predictions_gdf['trips_per_day']) / (predictions_gdf['ALAND'] / 2_589_988.11)

# Plot bike trip predictions with automatic legend for the ride counts
predictions_gdf.plot(column='trips_per_area', cmap='YlGnBu', legend=True, 
                legend_kwds={'label': "Predicted Number of Bike Trips"},
                edgecolor='black', linewidth=0.5, ax=ax)

# positive_predictions_gdf['trips_per_area'] = positive_predictions_gdf['trips_per_day'] / (positive_predictions_gdf['ALAND'] / 2_589_988.11)
# positive_predictions_gdf.plot(column='trips_per_area', cmap='YlGnBu', legend=True, 
#                 legend_kwds={'label': "Predicted Number of Bike Trips"},
#                 edgecolor='black', linewidth=0.5, ax=ax)

# Add title and remove axes for clarity
ax.set_title("BikeShare predictions", fontsize=14)
ax.set_axis_off()

# Plot major roads and parks
stl_secondary_roads.plot(ax=ax, color='whitesmoke', linewidth=0.5, alpha=1.0, label='_nolabel_')
stl_parks.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.7, label='_nolabel_')

# Create custom legend handles
bike_legend_patch = mpatches.Patch(color='gray', label='St. Louis City Parks')
roads_legend_line = mlines.Line2D([], [], color='lightgrey', linewidth=1.5, label='Major Roads')

# Add custom legend for parks and roads separately
ax.legend(handles=[bike_legend_patch, roads_legend_line], loc='upper right', title="Map Features")

# Set title and turn off axis
plt.axis('off')

In [ ]:
from matplotlib.colors import ListedColormap
import mapclassify

fig, ax = plt.subplots(figsize=(10, 8))
bincount = 10
# Classify data into quartiles explicitly
classifier = mapclassify.Quantiles(predictions_gdf['trips_per_area'], k=bincount)
predictions_gdf['quartile'] = classifier.yb

# Define color map
cmap = plt.cm.YlGnBu
# Evenly spaced numbers from 0 to 1
numbers = np.logspace(0, 1, bincount)
numbers = (numbers - 1) / 9
#numbers = np.linspace(0,1,10)
colors = cmap(numbers)  # 4 colors for quartiles
custom_cmap = ListedColormap(colors)

# Plot quartile data
predictions_gdf.plot(
    column='quartile',
    cmap=custom_cmap,
    alpha=0.9,
    edgecolor='black',
    linewidth=0.1,
    ax=ax,
    legend=False
)

# # Plot roads and parks
stl_secondary_roads.plot(ax=ax, color='whitesmoke', linewidth=0.5, alpha=1.0)
stl_parks.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.7)

# Custom legend for quartiles
quartile_labels = []
bins = classifier.bins
quartile_labels.append(f'{predictions_gdf["trips_per_area"].min():.2f} – {bins[0]:.2f}')
for i in range(1, len(bins)):
    quartile_labels.append(f'{bins[i-1]:.2f} – {bins[i]:.2f}')

quartile_patches = [
    mpatches.Patch(color=colors[i], label=quartile_labels[i]) for i in range(bincount)
]

# # Custom legend for map features
parks_patch = mpatches.Patch(color='gray', label='St. Louis City Parks')
roads_line = mlines.Line2D([], [], color='whitesmoke', linewidth=1.5, label='Major Roads')
parks_patch.set_zorder(9998)
roads_line.set_zorder(9998)

# # Add legends separately with clear distinction
leg1 = ax.legend(
    handles=quartile_patches,
    title="Density Quantiles \n (rides/day/sq.mi)",
    loc='upper left',
    facecolor='lightgray',  # or use fancybox=True
    framealpha=1.0
)
# Force legend on top
leg1.set_zorder(9999)
ax.add_artist(leg1)  # Add first legend explicitly back to plot

ax.legend(handles=[parks_patch, roads_line], title="Map Features", loc='upper right',
          facecolor='lightgray',   # Legend background
          )


# Set titles and remove axes
ax.set_title("Regression Predictions for Bike Share Use in St. Louis \n (Colored by Quantiles, 10 bins)", fontsize=12)
ax.set_axis_off()
cx.add_basemap(ax, crs=26915, source=cx.providers.CartoDB.Positron, zoom=12)

plt.show()

In [ ]:
# Step 1: Reload from disk (don't set CRS yet)
phase_1 = gpd.read_file('./phase_files/phase_1_v3.shp')
phase_2 = gpd.read_file('./phase_files/phase_2_v3.shp')
phase_3 = gpd.read_file('./phase_files/phase_3_v3.shp')



In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
bincount = 10

print(predictions_gdf.crs)
# print(predictions_gdf.crs)

# Classify into quantiles
classifier = mapclassify.Quantiles(predictions_gdf['trips_per_area'], k=bincount)
predictions_gdf['quartile'] = classifier.yb

# Simplify geometry for faster plotting
predictions_gdf['geometry'] = predictions_gdf.geometry.simplify(tolerance=10)

# Plot predictions
predictions_gdf.plot(
    column='quartile',
    cmap='YlGnBu',
    ax=ax,
    alpha=0.9,
    edgecolor='black',
    linewidth=0.1,
    legend=True
)

# Overlay Phase 2 zones
phase_1.plot(ax=ax, facecolor='white', edgecolor='green', linewidth=1, linestyle='--', alpha=0.5)
phase_2.plot(ax=ax, facecolor='white', edgecolor='blue', linewidth=1, linestyle='--', alpha=0.5)
phase_3.plot(ax=ax, facecolor='white', edgecolor='purple', linewidth=1, linestyle='--', alpha=0.5)



ax.set_axis_off()
plt.show()


In [ ]:
predictions_gdf['trips_per_day_plus'] = predictions_gdf['trips_per_day'].clip(lower=0) / 2
predictions_gdf['trips_per_area_plus'] = predictions_gdf['trips_per_day_plus'] / (predictions_gdf['ALAND'] / 2_589_988.11)
#predictions_gdf.loc[predictions_gdf['trips_per_area'] > 0, 'trips_per_area'].hist()
predictions_gdf['ALAND_sqmi'] = predictions_gdf['ALAND'] / 2_589_988.11
max_row = predictions_gdf[predictions_gdf['trips_per_area'] == predictions_gdf['trips_per_area'].max()]
print(max_row)
predictions_gdf['trips_per_area_plus'].hist(bins = 50)

In [ ]:
from matplotlib.colors import ListedColormap
import mapclassify

fig, ax = plt.subplots(figsize=(10, 8))
bincount = 10
# Classify data into quartiles explicitly
predictions_gdf_normal = predictions_gdf[predictions_gdf['trips_per_area_plus'] < 300]
classifier = mapclassify.EqualInterval(predictions_gdf_normal['trips_per_area_plus'], k=bincount)
predictions_gdf_normal['quartile'] = classifier.y

# Define color map
cmap = plt.cm.YlGnBu
# Evenly spaced numbers from 0 to 1
numbers = np.logspace(0, 1, bincount)
numbers = (numbers - 1) / 9
#numbers = np.linspace(0,1,10)
print(numbers)
colors = cmap(numbers)
custom_cmap = ListedColormap(colors)

# Plot quartile data
predictions_gdf_normal.plot(
    column='quartile',
    cmap=custom_cmap,
    alpha=0.9,
    edgecolor='black',
    linewidth=0.1,
    ax=ax,
    legend=False
)

# # Plot roads and parks
stl_secondary_roads.plot(ax=ax, color='whitesmoke', linewidth=0.5, alpha=1.0)
stl_parks.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.7)

# Custom legend for quartiles
actual_bins = len(classifier.bins)
quartile_labels = []
quartile_labels.append(f'{predictions_gdf_normal["trips_per_area_plus"].min():.2f} – {classifier.bins[0]:.2f}')
for i in range(1, actual_bins):
    quartile_labels.append(f'{classifier.bins[i-1]:.2f} – {classifier.bins[i]:.2f}')

quartile_patches = [
    mpatches.Patch(color=colors[i], label=quartile_labels[i]) for i in range(actual_bins)
]

# # Custom legend for map features
parks_patch = mpatches.Patch(color='gray', label='St. Louis City Parks')
roads_line = mlines.Line2D([], [], color='whitesmoke', linewidth=1.5, label='Major Roads')
parks_patch.set_zorder(9998)
roads_line.set_zorder(9998)

# # Add legends separately with clear distinction
leg1 = ax.legend(
    handles=quartile_patches,
    title="Density Quantiles \n (rides/day/sq.mi)",
    loc='upper left',
    facecolor='lightgray',  # or use fancybox=True
    framealpha=1.0
)
# Force legend on top
leg1.set_zorder(9999)
ax.add_artist(leg1)  # Add first legend explicitly back to plot

ax.legend(handles=[parks_patch, roads_line], title="Map Features", loc='upper right',
          facecolor='lightgray',   # Legend background
          )

# Set titles and remove axes
ax.set_title("Regression Predictions for Bike Share Use in St. Louis \n (Colored by Quantiles, 10 bins)", fontsize=12)
ax.set_axis_off()
cx.add_basemap(ax, crs=26915, source=cx.providers.CartoDB.Positron, zoom=12)


plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
import mapclassify

fig, ax = plt.subplots(figsize=(10, 8))

bincount = 10
# Classify data into quartiles explicitly
classifier = mapclassify.Quantiles(predictions_gdf['trips_per_area'], k=bincount)
predictions_gdf['quartile'] = classifier.yb

# Define color map
cmap = plt.cm.YlGnBu
# Evenly spaced numbers from 0 to 1
numbers = np.logspace(0, 1, bincount)
numbers = (numbers - 1) / 9
#numbers = np.linspace(0,1,10)
print(numbers)
colors = cmap(numbers)  # 4 colors for quartiles
custom_cmap = ListedColormap(colors)

# Plot quartile data
predictions_gdf.plot(
    column='quartile',
    cmap=custom_cmap,
    #alpha=0.9,
    edgecolor='black',
    linewidth=0.1,
    ax=ax,
    legend=False
)
print(predictions_gdf.total_bounds)

# # Plot roads and parks
stl_secondary_roads.plot(ax=ax, color='whitesmoke', linewidth=0.5, alpha=1.0)
stl_parks.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.7)

# Custom legend for quartiles
quartile_labels = []
bins = classifier.bins
quartile_labels.append(f'{predictions_gdf["trips_per_area"].min():.2f} – {bins[0]:.2f}')
for i in range(1, len(bins)):
    quartile_labels.append(f'{bins[i-1]:.2f} – {bins[i]:.2f}')

quartile_patches = [
    mpatches.Patch(color=colors[i], label=quartile_labels[i]) for i in range(bincount)
]

# # Custom legend for map features
parks_patch = mpatches.Patch(color='gray', label='St. Louis City Parks')
roads_line = mlines.Line2D([], [], color='whitesmoke', linewidth=1.5, label='Major Roads')
parks_patch.set_zorder(9998)
roads_line.set_zorder(9998)

# # Add legends separately with clear distinction
leg1 = ax.legend(
    handles=quartile_patches,
    title="Density Quantiles \n (people/sq.mi)",
    loc='center left',
    facecolor='lightgray',  # or use fancybox=True
    framealpha=1.0
)
# # Force legend on top
leg1.set_zorder(9999)
ax.add_artist(leg1)  # Add first legend explicitly back to plot

ax.legend(handles=[parks_patch, roads_line], title="Map Features", loc='upper left',
          facecolor='lightgray',   # Legend background
          )

# Set titles and remove axes
ax.set_title("Regression Predictions for Bike Share Use in St. Louis \n (Colored by Quantiles, 10 bins)", fontsize=12)
cx.add_basemap(ax, crs=26915, source=cx.providers.CartoDB.Positron, zoom=12)
#cx.add_basemap(ax, crs=26915, source=cx.providers.CartoDB.PositronOnlyLabels, zoom=11)

xmin, xmax = 710566.21999291, 746286.67716932  
ymin, ymax = 4267000.54757311, 4295000

ax.set_xlim([xmin, xmax])
ax.set_ylim([ymin, ymax])
ax.set_axis_off()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

bincount = 10
# Classify the prediction values into quantiles.
classifier = mapclassify.Quantiles(predictions_gdf['trips_per_area'], k=bincount)
predictions_gdf['quartile'] = classifier.yb

# Define colormap
cmap = plt.cm.YlGnBu
# Evenly spaced numbers from 0 to 1 (using logspace then scaling, as before)
numbers = np.logspace(0, 1, bincount)
numbers = (numbers - 1) / 9
colors = cmap(numbers)
custom_cmap = ListedColormap(colors)

# Plot the predictions layer (using the quantile classification)
predictions_gdf.plot(
    column='quartile',
    cmap=custom_cmap,
    edgecolor='black',
    linewidth=0.1,
    ax=ax,
    legend=False
)

# Plot roads and parks
stl_secondary_roads.plot(ax=ax, color='whitesmoke', linewidth=0.5, alpha=1.0)
stl_parks.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.7)

# Instead of filling with white, use no fill to let the prediction layer show through.
phase_1.plot(ax=ax, facecolor='white', edgecolor='green', linewidth=3, alpha=0.8)
phase_2.plot(ax=ax, facecolor='white', edgecolor='blue', linewidth=3, linestyle='dashdot', alpha=0.8)
phase_3.plot(ax=ax, facecolor='none', edgecolor='purple', linewidth=3, linestyle='dashed', alpha = 0.8)

# Create a custom legend for the predictions quantiles.
quartile_labels = []
bins = classifier.bins
# First label: from minimum to the first bin
quartile_labels.append(f'{predictions_gdf["trips_per_area"].min():.2f} – {bins[0]:.2f}')
for i in range(1, len(bins)):
    quartile_labels.append(f'{bins[i-1]:.2f} – {bins[i]:.2f}')
quartile_patches = [mpatches.Patch(color=colors[i], label=quartile_labels[i]) 
                     for i in range(bincount)]

# Create patches for the phases.
phase1_patch = mpatches.Patch(facecolor='white', edgecolor='green', linewidth=2, label='Rollout Phase 1', alpha = 0.8)
phase2_patch = mpatches.Patch(facecolor='white', edgecolor='blue', linewidth=2, linestyle='dashdot', label='Rollout Phase 2', alpha = 0.8)
phase3_patch = mpatches.Patch(facecolor='none', edgecolor='purple', linewidth=2, linestyle='dashed', label='Rollout Phase 3', alpha = 0.8)

# Create patches for parks and roads.
parks_patch = mpatches.Patch(color='gray', label='St. Louis City Parks')
roads_line = mlines.Line2D([], [], color='whitesmoke', linewidth=1.5, label='Major Roads')

# Add the quantile legend first.
leg1 = ax.legend(handles=quartile_patches,
                 title="Density Quantiles \n (people/sq.mi)",
                 loc='center left',
                 facecolor='lightgray', framealpha=1.0)
leg1.set_zorder(9999)
ax.add_artist(leg1)

# Add a separate legend for map features (roads, parks, and phases).
# Combine the phase patches with the other map feature patches.
feature_handles = [parks_patch, roads_line, phase1_patch, phase2_patch, phase3_patch]
leg2 = ax.legend(handles=feature_handles,
                 title="Map Features",
                 loc='upper left',
                 facecolor='lightgray',
                 framealpha=1.0)
leg2.set_zorder(9999)

# Set title, basemap, and extents.
ax.set_title("Regression Predictions for Bike Share Use in St. Louis \n (Colored by Quantiles, 10 bins)", fontsize=12)
import contextily as cx
cx.add_basemap(ax, crs=26915, source=cx.providers.CartoDB.Positron, zoom=12)

# Set custom extent (if needed)
xmin, xmax = 720566.22, 746286.68  
ymin, ymax = 4267000.55, 4295000
ax.set_xlim([xmin, xmax])
ax.set_ylim([ymin, ymax])
ax.set_axis_off()

plt.show()


In [ ]:
merged_gdf['density'] = merged_gdf['S0801_C02_049E'] / merged_gdf['ALAND']
#merged_gdf['density'] = merged_gdf['S0601_C02_047E'] / merged_gdf['ALAND']
#merged_gdf['density'] = merged_gdf['S0802_C01_07E'] / merged_gdf['ALAND'] S0601_C02_047E

merged_gdf['density'] = merged_gdf['S0101_C05_020E'] / merged_gdf['ALAND']
merged_gdf['density'] = merged_gdf['S0802_C01_078E'] / merged_gdf['ALAND']

merged_gdf['density'] = merged_gdf['S0802_C04_039E'] + merged_gdf['S0802_C04_039E']
merged_gdf['density'] = merged_gdf['S0801_C02_049E'] / merged_gdf['ALAND']
merged_gdf['density'] = merged_gdf['S0101_C05_004E'] / merged_gdf['ALAND']

#merged_gdf['density'] = merged_gdf['DP03_0062E'] # median household income

merged_gdf['density'] = (merged_gdf['DP03_0053E'] + merged_gdf['DP03_0054E'] + merged_gdf['DP03_0055E'] + merged_gdf['DP03_0056E']) / (merged_gdf['ALAND'] / 2_589_988.11)
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

#Plot bike trip predictions with automatic legend for the ride counts
merged_gdf.plot(column='density', cmap='YlGnBu', legend=True, 
                legend_kwds={'label': "Number of Households Per Square Mile"},
                edgecolor='black', linewidth=0.5, ax=ax)

# Add title and remove axes for clarity
ax.set_title("Density of Households with Income Between 10,000 and 49,999 Dollars", fontsize=14)
ax.set_axis_off()

# Plot major roads and parks
stl_secondary_roads.plot(ax=ax, color='whitesmoke', linewidth=0.5, alpha=1.0, label='_nolabel_')
stl_parks.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.7, label='_nolabel_')

# Create custom legend handles
bike_legend_patch = mpatches.Patch(color='gray', label='St. Louis City Parks')
roads_legend_line = mlines.Line2D([], [], color='lightgrey', linewidth=1.5, label='Major Roads')

# Add custom legend for parks and roads separately
ax.legend(handles=[bike_legend_patch, roads_legend_line], loc='upper right', title="Map Features")

# Set title and turn off axis
plt.axis('off')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

bincount = 10
# Classify data into quartiles explicitly
merged_gdf['density'] = merged_gdf['DP03_0062E']
classifier = mapclassify.Quantiles(merged_gdf['density'], k=bincount)
merged_gdf['quartile'] = classifier.yb

# Define color map
cmap = plt.cm.PiYG
# Evenly spaced numbers from 0 to 1
numbers = np.logspace(0, 1, bincount)
numbers = (numbers - 1) / 9
#numbers = np.linspace(0,1,10)
print(numbers)
colors = cmap(numbers)  # 4 colors for quartiles
custom_cmap = ListedColormap(colors)

# Plot quartile data
merged_gdf.plot(
    column='quartile',
    cmap=custom_cmap,
    alpha=0.9,
    edgecolor='black',
    linewidth=0.1,
    ax=ax,
    legend=False
)
print(merged_gdf.total_bounds)

# # Plot roads and parks
stl_secondary_roads.plot(ax=ax, color='whitesmoke', linewidth=0.5, alpha=1.0)
stl_parks.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.7)

# Custom legend for quartiles
quartile_labels = []
bins = classifier.bins
quartile_labels.append(f'{merged_gdf["density"].min():.2f} – {bins[0]:.2f}')
for i in range(1, len(bins)):
    quartile_labels.append(f'{bins[i-1]:.2f} – {bins[i]:.2f}')

quartile_patches = [
    mpatches.Patch(color=colors[i], label=quartile_labels[i]) for i in range(bincount)
]

# # Custom legend for map features
parks_patch = mpatches.Patch(color='gray', label='St. Louis City Parks')
roads_line = mlines.Line2D([], [], color='whitesmoke', linewidth=1.5, label='Major Roads')
parks_patch.set_zorder(9998)
roads_line.set_zorder(9998)

# # Add legends separately with clear distinction
leg1 = ax.legend(
    handles=quartile_patches,
    title="Density Quantiles \n (people/sq.mi)",
    loc='upper left',
    facecolor='lightgray',  # or use fancybox=True
    framealpha=1.0
)
# Force legend on top
leg1.set_zorder(9999)
ax.add_artist(leg1)  # Add first legend explicitly back to plot

ax.legend(handles=[parks_patch, roads_line], title="Map Features", loc='upper right',
          facecolor='lightgray',   # Legend background
          )

# Set titles and remove axes
ax.set_title("Median Household Income by Quantiles (10 bins)", fontsize=12)
cx.add_basemap(ax, crs=26915, source=cx.providers.CartoDB.Positron, zoom=12)
cx.add_basemap(ax, crs=26915, source=cx.providers.CartoDB.PositronOnlyLabels, zoom=11)

# xmin, xmax = 716566.21999291, 750286.67716932  
# ymin, ymax = 4268000.54757311, 4295000

# ax.set_xlim([xmin, xmax])
# ax.set_ylim([ymin, ymax])


plt.show()

In [ ]:
from matplotlib.colors import ListedColormap
import mapclassify

merged_gdf['density'] = merged_gdf['S0801_C02_049E'] / (merged_gdf['ALAND'] / 2_589_988.11) 

fig, ax = plt.subplots(figsize=(10, 8))

bincount = 10
# Classify data into quartiles explicitly
classifier = mapclassify.Quantiles(merged_gdf['density'], k=bincount)
merged_gdf['quartile'] = classifier.yb

# Define color map
cmap = plt.cm.YlGnBu
# Evenly spaced numbers from 0 to 1
numbers = np.logspace(0, 1, bincount)
numbers = (numbers - 1) / 9
#numbers = np.linspace(0,1,10)
colors = cmap(numbers)  # 4 colors for quartiles
custom_cmap = ListedColormap(colors)

# Plot quartile data
merged_gdf.plot(
    column='quartile',
    cmap=custom_cmap,
    edgecolor='black',
    linewidth=0.1,
    ax=ax,
    legend=False
)

# Plot roads and parks
stl_secondary_roads.plot(ax=ax, color='whitesmoke', linewidth=0.5, alpha=1.0)
stl_parks.plot(ax=ax, color='gray', linewidth=0.5, alpha=0.7)

# Custom legend for quartiles
quartile_labels = []
bins = classifier.bins
quartile_labels.append(f'{merged_gdf["density"].min():.2f} – {bins[0]:.2f}')
for i in range(1, len(bins)):
    quartile_labels.append(f'{bins[i-1]:.2f} – {bins[i]:.2f}')

quartile_patches = [
    mpatches.Patch(color=colors[i], label=quartile_labels[i]) for i in range(bincount)
]

# Custom legend for map features
parks_patch = mpatches.Patch(color='gray', label='St. Louis City Parks')
roads_line = mlines.Line2D([], [], color='whitesmoke', linewidth=1.5, label='Major Roads')

# Add legends separately with clear distinction
leg1 = ax.legend(handles=quartile_patches, title="Density Quantiles \n (people/sq.mi)", loc='upper left', facecolor='lightgray',)
ax.add_artist(leg1)  # Add first legend explicitly back to plot

ax.legend(handles=[parks_patch, roads_line], title="Map Features", loc='upper right', facecolor='lightgray',)
leg1.set_zorder(9999)

# Set titles and remove axes
ax.set_title("Density of Workers Living in a Household with \n One Vehicle Available (Quantiles, 10 bins)", fontsize=14)
ax.set_axis_off()

cx.add_basemap(ax, crs=26915, source=cx.providers.CartoDB.Positron, zoom=12)
#cx.add_basemap(ax, crs=26915, source=cx.providers.CartoDB.PositronOnlyLabels, zoom=11)

plt.show()